# Programmieraufgabe 4

Nach dem wir in der letzten Aufgabe schon verschiedene Einschrittverfahren verglichen haben, werden wir das dieses mal mit Butcher-Tabellen systematisieren und den Orbit einer Rakte auf einer Umlaufbahn um die Erde und den Mond berechnen.

Tragen Sie zunächst in der folgenen Zelle Ihren Namen ein:

In [ ]:
# Numerik gewöhnlicher Differentialgleichungen
# Sommersemester 2026
# Übungsblatt 7 - Programmieraufgabe 4
#
# [Nachname], [Vorname]
# [Vorname.Nachname@uni-a.de]

In [ ]:
import numpy as np
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import matplotlib.animation as animation

import time

from IPython.display import HTML

from util.plotting_04 import plot_RK_solution, init_animation, draw_frame
%matplotlib inline

Implementieren Sie einen Schritt des expliziten Runge-Kutta-Verfahren (Algorithmus 4.7). Wir stellen eine Butcher-Tabelle $\,\begin{array}{c|c}c & A \\ \hline & b^T\end{array}\,$ dabei durch ein Tupel `(A, b, c)` dar.

Achten Sie ausserdem wie in Programmieraufgabe 3 wieder darauf, dass es auch möglich sein soll, ein System von gewöhnlichen Differentialgleichungen zu lösen, also `y` als Array zu übergeben.

In [ ]:
def RK(f, t, y, h, Butcher):
    '''
        Berechnet mit dem expliziten Runge-Kutta-Verfahren mit 
        gegebener  Butcher Tabelle den nächsten Punkt der 
        Differentialgleichung y'(t) = f(t, y(t))
        Parameter:
            f       : rechte Seite der Differentialgleichung
            t       : aktueller Zeitpunkt
            y       : aktueller Funktionswert
            h       : Schrittweite
            Butcher : Butcher-Tabelle (A, b, c)
        Rückgabewert:
            Approximation von y(t + h)
    '''
    A, b, c = Butcher
    m = b.shape[0]
    if isinstance(y, (int, float)):
        n = 1
    else:
        n = y.shape[0]
    k = np.zeros((n,m))
    k[:,0] = f(t, y)
    for i in range(1, m):
        k[:,i] = ???
    ???
    return y + step

Als nächstes benötigen wir noch konkrete Werte für `A, b, c`. Verwenden Sie dazu das klassische (4-stufige) Runge-Kutta-Verfahren mit der Butcher-Tabelle 
$$
\begin{array}{c|cccc}
    0 & & & & \\ 
    \frac{1}{2} & \frac{1}{2} & & & \\
    \frac{1}{2} & 0 & \frac{1}{2} & & \\
    1 & 0 & 0 & 1 \\
    \hline & \frac{1}{6} & \frac{1}{3} & \frac{1}{3} & \frac{1}{6}
    \end{array}
$$

In [ ]:
def Butcher_RK4():
    '''
        Rückgabewert:
            A, b, c: Butcher-Tabelle für das klassische Runge-Kutta Verfahren

    '''
    A = np.zeros((4,4))
    A[1,0] = ???
    ???
    b = np.array(???)
    c = ???
    return A, b, c

Wie Sie bisher können Sie die nächste Zelle als formalen Test verwenden, ob Ihre Funktionen die richtigen Argumente akzeptieren und die korrekten Ausgaben liefern, vorallem für vektorwertiges `y`.

Beachten Sie wie immer, dass das nur eine Hilfestellung ist und ihre Funktionen auch Fehler enthalten können, die diese Tests nicht finden.

In [ ]:
f = lambda t, y : t - y
t = 0
y = 0
h = 1
RK(f, t, y, h, Butcher_RK4())
assert np.array_equal(RK(f, t, y, h, Butcher_RK4()), np.array([0.375])), "Falscher Wert für das Runge-Kutta-Verfahren"

# Vektorwertiges y
f_2 = lambda t, y : np.array([t - y[0], t - y[1]])
y_2 = np.array([0.0, 1.0])
assert np.array_equal(RK(f_2, t, y_2, h, Butcher_RK4()), np.array([0.375, 0.75])), "Falscher Wert für das Runge-Kutta-Verfahren mit vektorwertigem Input"


Wir können unsere Funktionen jetzt verwenden, um die Bahn einer Raumsonde zwischen Erde und Mond zu berechnen. Die Gravitationskraft, die auf die Rakete wirkt, wird durch die folgende Funktion beschrieben. 

Sie können diese Funktion einfach als gegebenes Modell verwenden. Die Details der Modellierung sind für die weitere Aufgabe nicht nötig, aber vielleicht trotzdem interessant: Die Einheit für die räumlichen Abstände ist hier die Distanz Erde-Mond, die zeitliche Einheit ist 1 Monat und das Koordinatensystem hat als Ursprung den gemeinsamen Schwerpunkt von Erde und Mond und rotiert mit dem Mond um diesen. Das heißt, sowohl Erde als auch Mond haben eine fixe Position, und nur die Raumsonde bewegt sich durch das System.

In [ ]:
def Erde_Mond_Rakete(t, y, mu):
    '''
        Rechte Seite der Differentialgleichung, die die Bahn 
        einer Raumsonde zwischen Erde und Mond beschreibt
        Parameter:
            t     : aktueller Zeitpunkt
            y     : aktueller Funktionswert (x_1, x'_1, x_2, x'_2)
            mu    : relative Masse des Mondes im Vergleich zur Masse Erde+Mond
        Rückgabewert:
            y_new : Wert der Funktion
    '''
    
    mu_hat = 1.0 - mu # relative Masse der Erde
    n1 =  ((y[0] + mu    )**2 + y[2] * y[2])**1.5
    n2 =  ((y[0] - mu_hat)**2 + y[2] * y[2])**1.5

    y_new = np.array([y[1], y[0], y[3], y[2]])
    y_new[1] += 2 * y[3] - mu_hat * (y[0] + mu) / n1 - mu * (y[0] - mu_hat) / n2
    y_new[3] -= 2 * y[1] + mu_hat *  y[2]       / n1 + mu *  y[2]           / n2
    return y_new

Vergleichen Sie jetzt verschiedene Schrittweiten (bzw. bei gegebenem Zeitinterval $[0, T]$ äquivalent verschiedene Schrittzahlen `n_steps`). Beschreiben sie kurz, was Sie sehen.

In [ ]:
mu = 0.012277470841006752  # relative Masse des Mondes
f = lambda t, u: Erde_Mond_Rakete(t, u, mu)

# Anfangswert
y0 = np.array([1.2, 0.0, 0.0, -1.04935750983035])
T = 6.1917317137
# Ein anderer interessanter Orbit ergibt sich aus folgender Startposition
# y0 = np.array([0.994, 0.0, 0.0, -2.001585106])
# T = 17.0652166

fig = plt.figure(figsize=(15,8))
for i, n_steps in enumerate([100, 500, 1000, 5000, 20000, 30000]):
    # Positionen und Geschwindigkeiten
    y = np.zeros((n_steps+1, 4))
    y[0,:] = y0
    # Beschleunigung
    a = np.zeros((n_steps+1, 2))
    a[0,:] = f(0, y0)[[1, 3]]
    # Berechnen der Bahn
    h = T / n_steps
    ts = np.linspace(0, T, n_steps + 1)
    for j, t in enumerate(ts[:-1]):
        y[j+1,:] = ???
        a[j+1,:] = f(t, y[j+1,:])[[1,3]]
    # Plotten
    plt.subplot(2, 3, i+1)
    plot_RK_solution(y, mu, xlims = (-1.5, 1.5), ylims=(-1.5, 1.5))

In [ ]:
# Ab welcher Schrittzahl sieht das Ergebnis optisch korrekt aus? 
#
# An welcher Position treten davor die meisten numerischen Fehler auf? Warum dort?
#
#

Abschließend können wir den Flug der Rakete auch noch animieren. Das Erstellen der Animation kann dabei einige Momente dauern.

In [ ]:
n_steps = y.shape[0] - 1
steps_per_frame = 100
frames = n_steps // steps_per_frame

u = y[:,[0,2]]
v = y[:,[1,3]]

plt.ioff()
fig = plt.figure(figsize=(9,9))
plt.title(f'Orbit  mit  klassischem Runge-Kutta-Verfahren, {n_steps} Schritte')
init_func = lambda: init_animation(mu, u[0,:], v[0,:], a[0,:])
# v und a werden für das Plotten skaliert, damit die Pfeile nicht zu lang werden
func = lambda frame: draw_frame(frame, steps_per_frame, u, 0.2*v, 0.2*a)
anim = animation.FuncAnimation(fig=fig, func=func, frames=frames, init_func=init_func, interval=50)
plt.ion()

HTML(anim.to_jshtml())